In [1]:
import os
import load_dotenv
from load_dotenv import load_dotenv

# This function will load all the variable from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv()

if os.environ.get("CLAUDE_API_KEY"):
    print("API KEY variable has been loaded")
else:
    raise ValueError("CLAUDE API KEY not found")

API KEY variable has been loaded


In [2]:
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-anthropic': '1.5.3'}}, output_version=None, model='vertex_ai.anthropic.claude-opus-4-6', max_tokens=4096, temperature=0.0, anthropic_api_url='https://genai-sharedservice-americas.pwcinternal.com', anthropic_api_key=SecretStr('**********'), anthropic_proxy=None, model_kwargs={})

In [3]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_anthropic.with_structured_output(llm_schema)

# llm_structured_output.invoke("The movie was great, what do you say ?")

In [ ]:
result = llm_structured_output.invoke("This movie is good")
result.model_dump()
result.model_dump()['movie_summary_flag']

#### **Conditional Chains**

In [8]:
# TASK - 1 [Prompt]
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative: {input}")
])

In [7]:
# TASK - 2 [LLM]
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-anthropic': '1.5.3'}}, output_version=None, model='vertex_ai.anthropic.claude-opus-4-6', max_tokens=4096, temperature=0.0, anthropic_api_url='https://genai-sharedservice-americas.pwcinternal.com', anthropic_api_key=SecretStr('**********'), anthropic_proxy=None, model_kwargs={})

In [6]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)-> str:

    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

#### Conditional Chain 1

In [9]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {input}")
])

# TASK - 2 [LLM]
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

# TASK - 3 [String Parser]
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

chain_linkedIn = linkedin_prompt | llm_anthropic | str_parser

#### Conditional Chain 2

In [ ]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human", "Create a post for the following text for LinkedIn: {text}")
    ])

    # TASK - 2 [LLM]

    llm_anthropic = ChatAnthropic(
        model=os.environ.get("CLAUDE_MODEL"), 
        temperature=0,
        api_key=os.environ.get("CLAUDE_API_KEY"),
        base_url=os.environ.get("CLAUDE_BASE_URL")
    )
    llm_anthropic

    # TASK - 3 [String Parser]

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_anthropic | str_parser

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)

#### Final Orchestration

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedIn),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [ ]:
final_orchestrator.invoke({"input": "I love this Stranger Things movie"})